# Prophet — Walmart Store Sales Forecasting

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ["WALMART_ROOT"] = "/content/drive/MyDrive/MLFinalAssignment"

In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("prophet", "mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'Prophet_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

In [ ]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import mlflow
from joblib import Parallel, delayed

from prophet import Prophet

from walmart_panel import WalmartPanel, PrecomputedForecaster
from walmart_prep import (
    FOLDS, HORIZON, WalmartFeatureBuilder, december_shape_report, load_raw,
    make_submission, score_fold, seasonal_naive, wmae, wmae_weights,
)

logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

print("tracking:", mlflow.get_tracking_uri())

train, test, features, stores = load_raw(str(ROOT))
SEED = 0

print(f"train {train.shape}   test {test.shape}")
print(f"train {train.Date.min().date()} .. {train.Date.max().date()}")
print(f"test  {test.Date.min().date()} .. {test.Date.max().date()}")

## `Prophet_Cleaning`

In [ ]:
MIN_OBS = 30

hol = features.loc[features.IsHoliday, ["Date"]].drop_duplicates().sort_values("Date")
HOLIDAY_NAME = {2: "super_bowl", 9: "labor_day", 11: "thanksgiving", 12: "christmas"}
holidays = pd.DataFrame({
    "holiday": hol.Date.dt.month.map(HOLIDAY_NAME),
    "ds": hol.Date,
    "lower_window": 0,
    "upper_window": 0,
})
pre = holidays[holidays.holiday == "christmas"].copy()
pre["holiday"] = "pre_christmas"
pre["ds"] = pre["ds"] - pd.Timedelta(weeks=1)
HOLIDAYS = pd.concat([holidays, pre], ignore_index=True).sort_values("ds").reset_index(drop=True)

with mlflow.start_run(run_name="Prophet_Cleaning"):
    mlflow.log_params({
        "series_model": "one Prophet per (Store, Dept), MAP fit, uncertainty_samples=0",
        "min_obs": MIN_OBS,
        "fallback": "seasonal naive via predict_frame for unfit series / missing cells",
        "holidays": ", ".join(sorted(HOLIDAYS.holiday.unique())),
        "weekly_grain": "ds = week-ending Friday; weekly/daily seasonality off, yearly on",
    })
    obs = train.groupby(["Store", "Dept"]).size()
    stats = {
        "n_series": int(len(obs)),
        "series_ge_min_obs": int((obs >= MIN_OBS).sum()),
        "series_lt_min_obs": int((obs < MIN_OBS).sum()),
        "median_obs_per_series": float(obs.median()),
        "n_holiday_rows": int(len(HOLIDAYS)),
    }
    mlflow.log_metrics(stats)

pd.Series(stats).to_frame("value")

## `Prophet_Baseline`

In [ ]:
baseline_scores = {}
with mlflow.start_run(run_name="Prophet_Baseline"):
    mlflow.log_param("model", "seasonal naive: lag-52, fallback pair median, then global median")
    for fold in FOLDS:
        tr = train[train.Date <= fold.cut]
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        s = score_fold(va.Weekly_Sales, seasonal_naive(tr, va), va)
        baseline_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])

BASELINE = {k: v["wmae"] for k, v in baseline_scores.items()}
pd.DataFrame(baseline_scores).round(1)

## `Prophet_CV`

In [ ]:
DEFAULT_CFG = dict(changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                   seasonality_mode="additive", use_holidays=True)


def _fit_one(i, ds, y, horizon_dates, cfg):
    """Fit one series; return (panel_row, yhat) with NaN on any failure."""
    import logging
    logging.getLogger("cmdstanpy").disabled = True
    mode = cfg["seasonality_mode"]
    if mode == "multiplicative" and (y <= 0).any():
        mode = "additive"
    try:
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=cfg["changepoint_prior_scale"],
                    seasonality_prior_scale=cfg["seasonality_prior_scale"],
                    seasonality_mode=mode,
                    holidays=HOLIDAYS if cfg["use_holidays"] else None,
                    uncertainty_samples=0)
        m.fit(pd.DataFrame({"ds": ds, "y": y}))
        out = m.predict(pd.DataFrame({"ds": horizon_dates}))["yhat"].to_numpy()
    except Exception:
        out = np.full(len(horizon_dates), np.nan)
    return i, out


def prophet_horizon(cutoff, cfg, keys=None, n_jobs=-1, verbose=5):
    """Fit the panel (or a subset of series) and return (panel, horizon_matrix)."""
    panel = WalmartPanel(train, features, stores, lookback=52).fit(cutoff)
    dates = pd.DatetimeIndex(panel.forecast_dates())
    tr = train[train.Date <= panel.cutoff_]
    horizon = np.full((panel.n_series_, panel.horizon), np.nan)
    jobs = []
    for (s, d), grp in tr.groupby(["Store", "Dept"]):
        key = (int(s), int(d))
        if keys is not None and key not in keys:
            continue
        i = panel.key_pos_.get(key)
        if i is None or len(grp) < MIN_OBS:
            continue
        jobs.append(delayed(_fit_one)(i, grp.Date.reset_index(drop=True),
                                      grp.Weekly_Sales.to_numpy(), dates, cfg))
    t0 = time.time()
    for i, out in Parallel(n_jobs=n_jobs, verbose=verbose)(jobs):
        horizon[i] = out
    n_ok = int(np.isfinite(horizon).all(axis=1).sum())
    print(f"fitted {n_ok}/{len(jobs)} series in {time.time() - t0:.0f}s "
          f"(panel has {panel.n_series_})")
    return panel, horizon


fold = next(f for f in FOLDS if f.name == "recent")
va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]

with mlflow.start_run(run_name="Prophet_CV"):
    mlflow.log_params(DEFAULT_CFG)
    mlflow.log_param("min_obs", MIN_OBS)
    panel_cv, horizon_cv = prophet_horizon(fold.cut, DEFAULT_CFG)
    y_hat = PrecomputedForecaster(panel_cv, horizon_cv)(va)
    s = score_fold(va.Weekly_Sales, y_hat, va)
    mlflow.log_metric("wmae_recent", s["wmae"])
    for k, v in s.items():
        if k.startswith("mae_"):
            mlflow.log_metric(f"{k}_recent", v)
    gain = 100 * (1 - s["wmae"] / BASELINE["recent"])
    mlflow.log_metric("gain_vs_naive_recent", gain)
    mlflow.log_metric("series_fitted", int(np.isfinite(horizon_cv).all(axis=1).sum()))
    print(f"\nrecent: naive {BASELINE['recent']:.1f} -> Prophet {s['wmae']:.1f}  ({gain:+.1f}%)")

CV_WMAE = s["wmae"]
pd.Series(s).round(1).to_frame("recent")

## `Prophet_Tuning`

In [ ]:
w_mass = (train.assign(w=wmae_weights(train.IsHoliday))
               .assign(ws=lambda d: d.w * d.Weekly_Sales.abs())
               .groupby(["Store", "Dept"]).ws.sum().sort_values(ascending=False))
SUBSET = set(map(lambda t: (int(t[0]), int(t[1])), w_mass.head(300).index))
va_sub = va.merge(pd.DataFrame(sorted(SUBSET), columns=["Store", "Dept"]),
                  on=["Store", "Dept"])
print(f"subset: {len(SUBSET)} series, {len(va_sub)} validation rows, "
      f"{100 * w_mass.head(300).sum() / w_mass.sum():.1f}% of holiday-weighted sales mass")

CONFIGS = [
    ("default", dict(DEFAULT_CFG)),
    ("flexible_trend", {**DEFAULT_CFG, "changepoint_prior_scale": 0.5}),
    ("multiplicative", {**DEFAULT_CFG, "seasonality_mode": "multiplicative"}),
    ("no_holidays", {**DEFAULT_CFG, "use_holidays": False}),
    ("tight_seasonality", {**DEFAULT_CFG, "seasonality_prior_scale": 1.0}),
    ("flex_multiplicative", {**DEFAULT_CFG, "changepoint_prior_scale": 0.5,
                             "seasonality_mode": "multiplicative"}),
]

results = []
with mlflow.start_run(run_name="Prophet_Tuning"):
    mlflow.log_params({"n_configs": len(CONFIGS), "subset_series": len(SUBSET),
                       "selection_fold": "recent", "selection_metric": "subset WMAE"})
    for name, cfg in CONFIGS:
        t0 = time.time()
        panel_s, horizon_s = prophet_horizon(fold.cut, cfg, keys=SUBSET, verbose=0)
        pred = PrecomputedForecaster(panel_s, horizon_s)(va_sub)
        w = wmae(va_sub.Weekly_Sales, pred, va_sub.IsHoliday)
        results.append({"name": name, **cfg, "subset_wmae": w})
        with mlflow.start_run(run_name=f"Prophet_Tuning__{name}", nested=True):
            mlflow.log_params(cfg)
            mlflow.log_metric("subset_wmae_recent", w)
            mlflow.log_metric("fit_seconds", time.time() - t0)
        print(f"{name:20s} subset WMAE={w:8.1f}  ({time.time() - t0:4.0f}s)")

    best = min(results, key=lambda r: r["subset_wmae"])
    BEST_CFG = {k: best[k] for k in DEFAULT_CFG}
    mlflow.log_params({f"best_{k}": v for k, v in BEST_CFG.items()})
    mlflow.log_metric("best_subset_wmae", best["subset_wmae"])

    if best["name"] == "default":
        BEST_RECENT_WMAE = CV_WMAE
        print(f"\nbest = default; reusing full-panel WMAE {BEST_RECENT_WMAE:.1f}")
    else:
        print(f"\nbest = {best['name']}; confirming with a full-panel pass...")
        panel_b, horizon_b = prophet_horizon(fold.cut, BEST_CFG)
        BEST_RECENT_WMAE = score_fold(
            va.Weekly_Sales, PrecomputedForecaster(panel_b, horizon_b)(va), va)["wmae"]
    mlflow.log_metric("best_wmae_recent", BEST_RECENT_WMAE)

print(f"\nbest config: {best['name']}  full-panel recent WMAE={BEST_RECENT_WMAE:.1f}  "
      f"naive={BASELINE['recent']:.1f}")
pd.DataFrame(results).round(1)

## `Prophet_Final`

In [ ]:
panel_full, horizon_full = prophet_horizon(None, BEST_CFG)
forecaster = PrecomputedForecaster(panel_full, horizon_full)

with mlflow.start_run(run_name="Prophet_Final") as final_run:
    mlflow.log_params(BEST_CFG)
    mlflow.log_param("trained_on", "all 143 weeks")
    mlflow.log_metric("cv_wmae_recent", BEST_RECENT_WMAE)

    y_pred = forecaster(test)
    print(f"predictions: n={len(y_pred)} mean={y_pred.mean():.1f} "
          f"min={y_pred.min():.1f} max={y_pred.max():.1f}")

    g = (WalmartFeatureBuilder(features, stores)
         .fit(train.drop(columns=["Weekly_Sales"]), train.Weekly_Sales).xmas_profile_)
    frame, verdict = december_shape_report(train, test, y_pred, g)
    print(frame.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"\npredicted peak {verdict['peak_predicted']} | implied {verdict['peak_implied']}")
    print("GATE:", "PASS" if verdict["passed"] else "REVIEW -- " + "; ".join(verdict["problems"]))
    mlflow.log_metric("december_xmas_gap_pct", verdict["xmas_gap_pct"])
    mlflow.log_metric("december_gate_passed", int(verdict["passed"]))

    logged_model = mlflow.pyfunc.log_model(
        name="model", python_model=forecaster,
        code_paths=[str(ROOT / "src" / "walmart_prep.py"),
                    str(ROOT / "src" / "walmart_panel.py")],
        input_example=test.head(3),
    )
    MODEL_URI = logged_model.model_uri

    sub = make_submission(test, y_pred, ROOT / "submissions" / "prophet.csv")
    mlflow.log_artifact(str(ROOT / "submissions" / "prophet.csv"))
    FINAL_RUN_ID = final_run.info.run_id

print(f"\nrun_id = {FINAL_RUN_ID}\nmodel_uri = {MODEL_URI}")
sub.head()

In [ ]:
from mlflow import MlflowClient

loaded = mlflow.pyfunc.load_model(MODEL_URI)
reloaded = np.asarray(loaded.predict(test))
print("max |reloaded - original| =", float(np.abs(reloaded - y_pred).max()))
assert np.allclose(reloaded, y_pred, atol=1e-3), "the logged pyfunc must reproduce its predictions"
print("pyfunc round-trip OK")

mv = mlflow.register_model(MODEL_URI, REGISTERED_MODEL_NAME)
print(f"registered {mv.name} version {mv.version}")
try:
    MlflowClient().set_registered_model_alias(mv.name, "prophet", mv.version)
    print(f"alias 'prophet' -> version {mv.version}")
except Exception as e:
    print("alias not set (optional):", str(e)[:80])

In [ ]:
print("experiments: https://dagshub.com/smama23/MLFinalProject/experiments")
print("registry   : https://dagshub.com/smama23/MLFinalProject/models")
try:
    runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
    cols = [c for c in ("tags.mlflow.runName", "metrics.wmae_recent", "metrics.cv_wmae_recent",
                        "metrics.december_gate_passed") if c in runs.columns]
    summary = runs[cols] if cols else runs
    try:
        display(summary)
    except NameError:
        print(summary.to_string(index=False))
except Exception as e:
    print("could not fetch run summary (model is already logged):", e)